In [111]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
class TitanicDataset(Dataset):
    def __init__(self, file_path,is_test=False):
        self.data = pd.read_csv(file_path)
        self.data['Sex'] = self.data['Sex'].map({'male': 0, 'female': 1})
        self.data['Age'] = self.data['Age'].fillna(self.data['Age'].median())
        self.data['Age'] = self.data['Age']/80
        self.data['Fare'] = self.data['Fare'].fillna(self.data['Fare'].median())
        self.data['Fare'] = self.data['Fare']/500

        self.data['Title'] = self.data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
        self.data['Title'] = self.data['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
        self.data['Title'] = self.data['Title'].replace('Mlle', 'Miss')
        self.data['Title'] = self.data['Title'].replace('Ms', 'Miss')
        self.data['Title'] = self.data['Title'].replace('Mme', 'Mrs')
       
        title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
        self.data['Title'] = self.data['Title'].map(title_mapping).fillna(0)
        features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Fare', 'Title']

        self.x_data = torch.tensor(self.data[features].values.astype(float), dtype=torch.float32)
        self.is_test = is_test
        if not self.is_test:
            self.y_data = torch.tensor(self.data['Survived'].values, dtype=torch.float32)
        else:
            self.y_data = torch.zeros(len(self.data))
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]
    



In [113]:
train_dataset = TitanicDataset('../titanic_data/train.csv')
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True,num_workers=0)

In [ ]:
class TitanicModel(torch.nn.Module):
    def __init__(self):
        super(TitanicModel, self).__init__()     
        self.fc1 = torch.nn.Linear(6, 32)
        self.fc2 = torch.nn.Linear(32, 16)
        self.fc3 = torch.nn.Linear(16, 1)
       
      

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.nn.functional.dropout(x, p=0.5, training=self.training)
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x
    
model = TitanicModel()    

In [125]:
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

if __name__ == "__main__":
    for epoch in range(20):
        for x_data, y_data in train_loader:
            outputs = model(x_data)
            loss = criterion(outputs.squeeze(), y_data)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f'Epoch [{epoch+1}/20], Loss: {loss.item():.4f}')

Epoch [1/20], Loss: 0.5375
Epoch [2/20], Loss: 0.2998
Epoch [3/20], Loss: 0.4027
Epoch [4/20], Loss: 0.3311
Epoch [5/20], Loss: 0.4267
Epoch [6/20], Loss: 0.2292
Epoch [7/20], Loss: 0.5692
Epoch [8/20], Loss: 0.3847
Epoch [9/20], Loss: 0.4001
Epoch [10/20], Loss: 0.4567
Epoch [11/20], Loss: 0.3290
Epoch [12/20], Loss: 0.6766
Epoch [13/20], Loss: 0.3599
Epoch [14/20], Loss: 0.3956
Epoch [15/20], Loss: 0.3256
Epoch [16/20], Loss: 0.3346
Epoch [17/20], Loss: 0.4399
Epoch [18/20], Loss: 0.2139
Epoch [19/20], Loss: 0.5488
Epoch [20/20], Loss: 0.2147


In [116]:
test_dataset = TitanicDataset('../titanic_data/test.csv',is_test=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)
model.eval()
predictions = []        
with torch.no_grad():
    for x_data, _ in test_loader:
        outputs = model(x_data)
        predicted = (outputs.squeeze() > 0.5).int()
        predictions.extend(predicted.cpu().numpy())

In [126]:
raw_test_df = pd.read_csv('../titanic_data/test.csv')

submission = pd.DataFrame({
    "PassengerId": raw_test_df["PassengerId"],
    "Survived": predictions
})

submission.to_csv('submission10.csv', index=False)
